Notebook 3 is where the real transformation happens. The data 
we cleaned in Notebook 2 is accurate but it cannot go directly 
into a machine learning model in its current form. Level 
variables like GDP and CPI trend upward forever creating 
spurious correlations that would teach the model mathematical 
coincidences rather than genuine economic relationships. This 
notebook fixes that through three specific transformations.

First we convert level variables into growth rates to achieve 
stationarity. Second we create lag features that give the model 
a memory of past economic conditions since what happened 6 months 
ago often matters more than what is happening right now. Third 
we create our target variables by shifting GDP growth, inflation, 
and unemployment forward 6 months so the model learns to predict 
where the economy is heading rather than where it currently is.

By the end of this notebook we will have a clean, stationary, 
feature rich dataset ready for machine learning.

Input: data/cleaned_fred_data.csv
Output: data/featured_fred_data.csv

Loading our cleaned data and importing the tools needed
for transformation.

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import StandardScaler

In [3]:
df = pd.read_csv('../data/cleaned_fred_data.csv',
                  index_col=0,
                  parse_dates=True)
print(f"Data loaded: {df.shape}")
print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}")

Data loaded: (412, 10)
Date range: 1992-01-31 to 2026-04-30


In [10]:
df.head()

,GDP,CPI,Unemployment,FedFunds,M2,YieldCurve,IndPro,RetailSales,Sentiment,JoblessClaims
1992-01-31,10236.435,138.3,7.3,4.03,3381.2,2.20,61.4616,159177.0,67.5,441000.0
1992-02-29,10236.435,138.6,7.4,4.06,3400.0,2.00,61.8955,159189.0,68.8,444000.0
1992-03-31,10236.435,139.1,7.4,3.98,3403.9,1.94,62.4232,158647.0,76.0,420000.0
1992-04-30,10347.429,139.4,7.4,3.73,3399.7,2.15,62.8970,159921.0,77.2,424000.0
1992-05-31,10347.429,139.7,7.6,3.82,3398.6,2.14,63.1040,160471.0,79.2,411000.0


Before transforming anything we formally test each 
variable for stationarity using the Augmented Dickey Fuller 
test. This tells us exactly which variables need transformation 
and which are already stationary.

In [9]:
def test_stationarity(series, name):
    result = adfuller(series.dropna())
    pvalue = result[1]
    is_stationary = pvalue < 0.05
    status = "STATIONARY" if is_stationary else "NOT STATIONARY"
    print(f"{name:<20} p-value: {pvalue:.4f} {status}")

print("Augmented Dickey Fuller Stationarity Test Results")
print("=" * 60)
print("Rule: p-value below 0.05 = stationary")
print("=" * 60)

for col in df.columns:
    test_stationarity(df[col], col)

Augmented Dickey Fuller Stationarity Test Results
Rule: p-value below 0.05 = stationary
GDP                  p-value: 0.9840 NOT STATIONARY
CPI                  p-value: 0.9986 NOT STATIONARY
Unemployment         p-value: 0.0301 STATIONARY
FedFunds             p-value: 0.0472 STATIONARY
M2                   p-value: 0.9938 NOT STATIONARY
YieldCurve           p-value: 0.0073 STATIONARY
IndPro               p-value: 0.0741 NOT STATIONARY
RetailSales          p-value: 0.9982 NOT STATIONARY
Sentiment            p-value: 0.5891 NOT STATIONARY
JoblessClaims        p-value: 0.0000 STATIONARY


Transform non stationary variables into stationary 
ones. Level variables become growth rates. This simultaneously 
achieves stationarity and creates more economically meaningful 
features. A 2% GDP growth rate is far more informative than 
a raw GDP value of 23 trillion dollars.

In [11]:
fe = pd.DataFrame(index=df.index)

# GDP -- year over year growth rate
fe['GDP_growth'] = df['GDP'].pct_change(4) * 100

# CPI -- year over year inflation rate
fe['Inflation_yoy'] = df['CPI'].pct_change(12) * 100

# Inflation month over month (captures sudden price changes)
fe['Inflation_mom'] = df['CPI'].pct_change(1) * 100

# Unemployment -- use level but add rate of change
fe['Unemployment'] = df['Unemployment']
fe['Unemp_change_3m'] = df['Unemployment'].diff(3)
fe['Unemp_change_12m'] = df['Unemployment'].diff(12)

# Federal Funds Rate -- level plus change
fe['FedFunds'] = df['FedFunds']
fe['FedFunds_change_6m'] = df['FedFunds'].diff(6)

# Yield Curve -- level plus binary inversion flag
fe['YieldCurve'] = df['YieldCurve']
fe['YieldCurve_change'] = df['YieldCurve'].diff(3)
fe['Yield_inverted'] = (df['YieldCurve'] < 0).astype(int)
b
# M2 Money Supply -- year over year growth
fe['M2_growth'] = df['M2'].pct_change(12) * 100

# Industrial Production -- year over year growth
fe['IndPro_growth'] = df['IndPro'].pct_change(12) * 100
fe['IndPro_growth_3m'] = df['IndPro'].pct_change(3) * 100

# Retail Sales -- year over year growth
fe['Retail_growth'] = df['RetailSales'].pct_change(12) * 100

# Consumer Sentiment -- level plus change
fe['Sentiment'] = df['Sentiment']
fe['Sentiment_change_6m'] = df['Sentiment'].diff(6)

# Jobless Claims -- year over year growth
fe['Claims_growth'] = df['JoblessClaims'].pct_change(12) * 100

print("Transformations complete")
print(f"Features created so far: {fe.shape[1]}")
print(fe.head())

Transformations complete
Features created so far: 18
            GDP_growth  Inflation_yoy  Inflation_mom  Unemployment  \
1992-01-31         NaN            NaN            NaN           7.3   
1992-02-29         NaN            NaN       0.216920           7.4   
1992-03-31         NaN            NaN       0.360750           7.4   
1992-04-30         NaN            NaN       0.215672           7.4   
1992-05-31    1.084303            NaN       0.215208           7.6   

            Unemp_change_3m  Unemp_change_12m  FedFunds  FedFunds_change_6m  \
1992-01-31              NaN               NaN      4.03                 NaN   
1992-02-29              NaN               NaN      4.06                 NaN   
1992-03-31              NaN               NaN      3.98                 NaN   
1992-04-30              0.1               NaN      3.73                 NaN   
1992-05-31              0.2               NaN      3.82                 NaN   

            YieldCurve  YieldCurve_change  Yield_in

In [12]:
print("Stationarity Test on Transformed Variables")
print("=" * 60)

for col in fe.columns:
    test_stationarity(fe[col], col)

Stationarity Test on Transformed Variables
GDP_growth           p-value: 0.0004 STATIONARY
Inflation_yoy        p-value: 0.0076 STATIONARY
Inflation_mom        p-value: 0.0031 STATIONARY
Unemployment         p-value: 0.0301 STATIONARY
Unemp_change_3m      p-value: 0.0000 STATIONARY
Unemp_change_12m     p-value: 0.0019 STATIONARY
FedFunds             p-value: 0.0472 STATIONARY
FedFunds_change_6m   p-value: 0.0001 STATIONARY
YieldCurve           p-value: 0.0073 STATIONARY
YieldCurve_change    p-value: 0.0016 STATIONARY
Yield_inverted       p-value: 0.0025 STATIONARY
M2_growth            p-value: 0.0020 STATIONARY
IndPro_growth        p-value: 0.0105 STATIONARY
IndPro_growth_3m     p-value: 0.0001 STATIONARY
Retail_growth        p-value: 0.0017 STATIONARY
Sentiment            p-value: 0.5891 NOT STATIONARY
Sentiment_change_6m  p-value: 0.0003 STATIONARY
Claims_growth        p-value: 0.0000 STATIONARY


Create lag features. Economic relationships do not 
happen instantly. When the Fed raises interest rates today 
the full effect on GDP takes 6 to 18 months to materialise. 
Lag features give our model a memory of past conditions so 
it can learn these delayed relationships from historical data.

In [13]:
cols_to_lag = [
    'GDP_growth', 'Inflation_yoy', 'Unemployment',
    'FedFunds', 'YieldCurve', 'M2_growth',
    'IndPro_growth', 'Retail_growth', 'Sentiment',
    'Claims_growth', 'Yield_inverted', 'FedFunds_change_6m',
    'Unemp_change_3m', 'Sentiment_change_6m'
]

lag_periods = [1, 3, 6, 12]

for col in cols_to_lag:
    for lag in lag_periods:
        fe[f'{col}_lag{lag}'] = fe[col].shift(lag)
        
print(f"Total features after lags: {fe.shape[1]}")

Total features after lags: 74


Create target variables. MacroSense predicts 6 months 
into the future. To teach the model this we shift our three 
target variables forward by 6 months. This means each row now 
contains current features alongside the actual economic outcome 
that happened 6 months later, exactly the relationship we 
want the model to learn.

In [16]:
fe['target_GDP'] = fe['GDP_growth'].shift(-6)
fe['target_Inflation'] = fe['Inflation_yoy'].shift(-6)
fe['target_Unemployment'] = fe['Unemployment'].shift(-6)

print("target_GDP      >     GDP growth rate 6 months from now")
print("target_Inflation    > Inflation rate 6 months from now")
print("target_Unemployment > Unemployment rate 6 months from now")

target_GDP      >     GDP growth rate 6 months from now
target_Inflation    > Inflation rate 6 months from now
target_Unemployment > Unemployment rate 6 months from now


In [17]:
fe_clean = fe.dropna()

print(f"Rows before dropping missing: {len(fe)}")
print(f"Rows after dropping missing:  {len(fe_clean)}")
print(f"Rows removed: {len(fe) - len(fe_clean)}")
print(f"\nFinal dataset shape: {fe_clean.shape}")
print(f"Date range: {fe_clean.index[0].date()} to {fe_clean.index[-1].date()}")
print(f"\nFeatures: {fe_clean.shape[1] - 3}")
print(f"Target variables: 3")

save_path = os.path.join('..', 'data', 'featured_fred_data.csv')
fe_clean.to_csv(save_path)
print(f"\nFeatured data saved to: data/featured_fred_data.csv")

Rows before dropping missing: 412
Rows after dropping missing:  382
Rows removed: 30

Final dataset shape: (382, 77)
Date range: 1994-01-31 to 2025-10-31

Features: 74
Target variables: 3

Featured data saved to: data/featured_fred_data.csv


In [21]:
fe_clean.head()

,GDP_growth,Inflation_yoy,Inflation_mom,Unemployment,Unemp_change_3m,Unemp_change_12m,FedFunds,FedFunds_change_6m,YieldCurve,YieldCurve_change,...,Unemp_change_3m_lag3,Unemp_change_3m_lag6,Unemp_change_3m_lag12,Sentiment_change_6m_lag1,Sentiment_change_6m_lag3,Sentiment_change_6m_lag6,Sentiment_change_6m_lag12,target_GDP,target_Inflation,target_Unemployment
1994-01-31,2.343711,2.450980,0.000000,6.6,-0.2,-0.7,3.05,-0.01,1.58,0.14,...,-0.1,-0.2,0.0,6.7,-2.9,-12.3,12.7,1.947689,2.698962,6.1
1994-02-28,0.970363,2.515723,0.273411,6.6,0.0,-0.5,3.25,0.22,1.48,-0.13,...,-0.2,-0.3,-0.3,17.3,0.9,-9.3,10.5,0.584585,2.900552,6.0
1994-03-31,0.970363,2.651779,0.272665,6.5,0.0,-0.5,3.34,0.25,1.56,-0.02,...,-0.2,-0.3,-0.4,15.9,6.7,-8.0,10.3,0.584585,2.965517,5.9
1994-04-30,2.338696,2.364395,0.067981,6.4,-0.2,-0.7,3.56,0.57,1.33,-0.25,...,-0.2,-0.1,-0.2,13.6,17.3,-2.9,12.3,1.736852,2.609890,5.8
1994-05-31,1.355183,2.288488,0.203804,6.1,-0.5,-1.0,4.01,0.99,1.16,-0.32,...,0.0,-0.2,0.0,9.9,15.9,0.9,-5.0,1.145570,2.602740,5.6


In [23]:
target_cols = ['target_GDP', 'target_Inflation', 'target_Unemployment']
feature_cols = [c for c in fe_clean.columns if c not in target_cols]

X = fe_clean[feature_cols]
y = fe_clean[target_cols]

split_point = int(len(fe_clean) * 0.8)

X_train = X.iloc[:split_point]
X_test  = X.iloc[split_point:]
y_train = y.iloc[:split_point]
y_test  = y.iloc[split_point:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

import joblib
joblib.dump(scaler, '../outputs/scaler.pkl')

print("Train test split complete")b
print(f"Training: {X_train.shape[0]} months ({X_train.index[0].date()} to {X_train.index[-1].date()})")
print(f"Testing:  {X_test.shape[0]} months ({X_test.index[0].date()} to {X_test.index[-1].date()})")
print(f"Features: {X_train.shape[1]}")
print(f"Scaler saved to outputs/scaler.pkl")

Train test split complete
Training: 305 months (1994-01-31 to 2019-05-31)
Testing:  77 months (2019-06-30 to 2025-10-31)
Features: 74
Scaler saved to outputs/scaler.pkl


In [24]:
print("Training set covers:")
print(f"  From: {X_train.index[0].date()}")
print(f"  To:   {X_train.index[-1].date()}")
print(f"  Rows: {len(X_train)}")

print("\nTest set covers:")
print(f"  From: {X_test.index[0].date()}")
print(f"  To:   {X_test.index[-1].date()}")
print(f"  Rows: {len(X_test)}")

print("\nSample of feature names:")
for i, col in enumerate(feature_cols[:10]):
    print(f"  {col}")
print(f"  ... and {len(feature_cols) - 10} more features")

print(f"\nTarget variables:")
for col in target_cols:
    print(f"  {col}")

Training set covers:
  From: 1994-01-31
  To:   2019-05-31
  Rows: 305

Test set covers:
  From: 2019-06-30
  To:   2025-10-31
  Rows: 77

Sample of feature names:
  GDP_growth
  Inflation_yoy
  Inflation_mom
  Unemployment
  Unemp_change_3m
  Unemp_change_12m
  FedFunds
  FedFunds_change_6m
  YieldCurve
  YieldCurve_change
  ... and 64 more features

Target variables:
  target_GDP
  target_Inflation
  target_Unemployment


Notebook 3 is done. The data has been completely transformed 
from raw FRED downloads into a machine learning ready dataset.

Level variables that were trending upward and creating spurious 
correlations are now expressed as growth rates and changes. The 
model will learn genuine economic relationships rather than 
mathematical coincidences driven by shared upward trends.

Lag features give the model a 12 month memory of past conditions 
across 14 key economic variables. The model can now learn that 
what the yield curve was doing 6 months ago matters for predicting 
GDP growth today.

Target variables are shifted 6 months forward so every row in 
our training data contains current economic conditions alongside 
the actual outcome that followed 6 months later. This is the 
fundamental structure that enables genuine economic forecasting.

The dataset is split chronologically with 80% for training and 
20% for testing. The model will learn from the past and be 
evaluated on genuinely unseen future data, the only honest 
way to evaluate a forecasting system.

Input: data/cleaned_fred_data.csv
Output: data/featured_fred_data.csv
        outputs/scaler.pkl

Next: Notebook 4 -- Model Building